<a href="https://colab.research.google.com/github/muhammadusmanshakir/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Section 1: Setup and Environment Check
import pandas as pd
import numpy as np
import sklearn
print(f"Environment ready. Pandas: {pd.__version__}, Scikit-Learn: {sklearn.__version__}")

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a **Random Forest Classifier** because the dataset is tabular with mixed numeric
signals (traffic, content, engagement) and the relationship between these signals and a
page declining is unlikely to be linear or additive. Random Forest captures interactions
between features without assuming linearity, handles the mix of scales reasonably well,
and gives feature importances I can use for interpretation.

The model is used as directional decision support, not as a causal model.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**First attempt (kept here for the record, not used as the final model):** my first pass
defined the modeling target as the exact same rule as my Week-4 baseline —
`(days_since_last_update >= 90) & (impressions_90d >= 1000)` — and trained a Random Forest
on it using `days_since_last_update` and `impressions_90d` as two of the four input
features. That model scored **1.0000 accuracy, 1.0000 F1, zero misclassifications**. A
perfect score is not a win — it is the leakage lesson from notebook 03 repeating itself:
the target was a deterministic function of two of its own inputs, so there was nothing for
the model to actually learn. I discarded this target for that reason.

**Final target:** `decline_target` = 1 when a page's impressions in the most recent 30-day
window (`impressions_last_30d`) are lower than the prior 30-day window
(`impressions_prev_30d`), else 0. This is a genuinely different question from the baseline
rule — "did this page's visibility fall" rather than "is this page old and visible" — and
none of the columns used to build the target (`impressions_last_30d`,
`impressions_prev_30d`) are included as model features.

**Split:** 80/20 train-test split, stratified on `decline_target`, `random_state=42`. This
produced 24,000 training rows and 6,000 test rows, with the ~65.7% decline rate preserved
in both splits.

**Features used (23):** search/content descriptors (`search_volume`, `competition`, `cpc`,
`word_count`, `char_count`), 90-day engagement and traffic (`impressions_90d`,
`clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`,
`ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`),
content metadata (`content_age_days`, `age_tier_order`, `days_since_last_update`), and
performance signals (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`,
`ai_traffic_pct`).

**Deliberately excluded:** `content_id`, `client_id` (identifiers, not predictive),
`trend_direction`, `trend_pct` (describe the observed trend and could leak the outcome),
`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
`clicks_prev_30d`, `sessions_prev_30d` (the exact windows used to build the target).

In [ ]:
# ============================================
# SECTION 2 — DATA LOADING, TARGET, SPLIT DESIGN
# ============================================

from sklearn.model_selection import train_test_split

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)
print(f"Rows loaded: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ------------------------------------------------
# Final, independent target
# ------------------------------------------------
# 1 = impressions fell from the prior 30-day window to the latest 30-day window
# 0 = otherwise
# Built ONLY from the two window columns below -- neither is used as a feature.

df["decline_target"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
).astype(int)

print("\nDecline target distribution:")
print(df["decline_target"].value_counts())
print("\nDecline target proportions:")
print(df["decline_target"].value_counts(normalize=True))

# ------------------------------------------------
# Feature selection (leak-safe)
# ------------------------------------------------

feature_columns = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

X = df[feature_columns].copy()
y = df["decline_target"].copy()

# Missing values: median-fill, after clearing any inf values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
print("\nMissing values remaining:", X.isna().sum().sum())

# ------------------------------------------------
# Leakage check
# ------------------------------------------------

forbidden_features = [
    "decline_target", "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "content_id", "client_id",
]
leaked = [c for c in X.columns if c in forbidden_features]
assert not leaked, f"Leakage detected: {leaked}"
print("Leakage check passed -- no forbidden columns in the feature set.")

# ------------------------------------------------
# Train-test split
# ------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("\nSplit design:")
print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

To compare fairly, I apply my **Week-4 baseline rule** (`days_since_last_update >= 90 AND
impressions_90d >= 1000` → predict "declining") to the exact same `X_test` rows and score
it against the exact same `y_test` labels, using the exact same metrics as the Random
Forest. The table below is generated from code, not typed in by hand.

In [ ]:
# ============================================
# SECTION 3 — MODEL TRAINING AND BASELINE COMPARISON
# ============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
)

# ------------------------------------------------
# Train Random Forest
# ------------------------------------------------

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

model_accuracy  = accuracy_score(y_test, y_pred)
model_f1        = f1_score(y_test, y_pred)
model_precision = precision_score(y_test, y_pred, zero_division=0)
model_recall    = recall_score(y_test, y_pred, zero_division=0)
model_auc       = roc_auc_score(y_test, y_prob)

# ------------------------------------------------
# Week-4 baseline rule, scored on the SAME test rows / SAME target
# ------------------------------------------------

baseline_pred = (
    (X_test["days_since_last_update"] >= 90) & (X_test["impressions_90d"] >= 1000)
).astype(int)

baseline_accuracy  = accuracy_score(y_test, baseline_pred)
baseline_f1        = f1_score(y_test, baseline_pred, zero_division=0)
baseline_precision = precision_score(y_test, baseline_pred, zero_division=0)
baseline_recall    = recall_score(y_test, baseline_pred, zero_division=0)

majority_class_rate = y_test.value_counts(normalize=True).max()

# ------------------------------------------------
# Comparison table (built from real numbers, not typed in)
# ------------------------------------------------

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "F1-score", "Precision", "Recall"],
    "Majority-class base rate": [majority_class_rate, np.nan, np.nan, np.nan],
    "Week-4 baseline rule": [baseline_accuracy, baseline_f1, baseline_precision, baseline_recall],
    "Week-5 Random Forest": [model_accuracy, model_f1, model_precision, model_recall],
})
comparison["Improvement (RF - baseline)"] = (
    comparison["Week-5 Random Forest"] - comparison["Week-4 baseline rule"]
)

print("MODEL vs BASELINE (same test split, same target, same metrics)")
print("=" * 65)
display(comparison.round(4))

print(f"\nRandom Forest ROC-AUC: {model_auc:.4f}")
print(f"Majority-class base rate: {majority_class_rate:.4f}")

print("\nCONFUSION MATRIX -- Random Forest")
print(confusion_matrix(y_test, y_pred))
print("\nCONFUSION MATRIX -- Week-4 baseline rule")
print(confusion_matrix(y_test, baseline_pred))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric
table.*

The comparison table above shows whether the Random Forest actually beats the Week-4 rule
on the same target and the same rows, rather than comparing two different problems. Accuracy
alone should still be read next to the majority-class base rate (~65.7% decline rate), since
a model that is only a little better than "always predict decline" is not doing much work.

Below I look at feature importance and the specific rows the model got wrong, since a big
metric table can hide a model that leans on one feature or fails in a specific segment.

In [ ]:
# ============================================
# SECTION 4 — FEATURE IMPORTANCE AND ERROR ANALYSIS
# ============================================

importances = pd.Series(
    model.feature_importances_, index=X.columns
).sort_values(ascending=False)

print("TOP 10 FEATURE IMPORTANCES")
print("=" * 45)
print(importances.head(10))

# ------------------------------------------------
# Misclassified examples
# ------------------------------------------------

error_mask = y_test != y_pred
errors = X_test.loc[error_mask].copy()
errors["actual"] = y_test.loc[error_mask]
errors["predicted"] = y_pred[error_mask]

print("\nNUMBER OF MISCLASSIFICATIONS")
print("=" * 45)
print(f"Errors: {len(errors):,}")
print(f"Error rate: {len(errors) / len(y_test):.4f}")

print("\nSAMPLE MISCLASSIFICATIONS")
display(errors.head(10))

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("\nERROR BREAKDOWN")
print("=" * 45)
print(f"True negatives : {tn:,}")
print(f"False positives: {fp:,}")
print(f"False negatives: {fn:,}")
print(f"True positives : {tp:,}")

The model relies most heavily on overall search visibility (`impressions_90d`), average
search position, and the number of days with recorded impressions -- signals about how much
sustained visibility a page has, rather than raw content attributes like word count. These
importances describe how the Random Forest split on the available features; they are not
causal claims about what *causes* a decline.

False positives (predicted declining, actually stable) and false negatives (predicted
stable, actually declining) both exist -- the model is not perfect, which is expected and
healthy: it means the target is not a deterministic function of the inputs the way my
first, discarded attempt was.

The target itself (`decline_target`) is a rule-derived proxy for "this page's visibility is
falling," not a human-verified judgment that a page needs a content refresh. So this model
should be read as decision-support for prioritizing review, not as proof that any specific
page needs intervention.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled -- markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.

In [ ]:
# ============================================
# FINAL SELF-CHECK
# ============================================

required_objects = [
    "df", "X", "y", "X_train", "X_test", "y_train", "y_test",
    "model", "y_pred", "model_accuracy", "model_f1", "model_auc",
    "baseline_pred", "baseline_accuracy", "baseline_f1", "comparison",
]

missing = [obj for obj in required_objects if obj not in globals()]
assert not missing, f"Missing objects: {missing}"

assert len(df) == 30000, "Unexpected dataset size."
assert len(X_train) + len(X_test) == len(df), "Train/test split does not cover the full dataset."

print("SELF-CHECK PASSED")
print("=" * 45)
print(f"Dataset rows: {len(df):,}")
print("Model: Random Forest Classifier (decline_target)")
print(f"Features: {len(feature_columns)}")
print("Train/test split: 80/20, stratified, random_state=42")
print("Baseline and model evaluated on the same test rows and same metrics.")